# Reinforcement Learning Agent for Battleship

Battleship is a classic two-player game where each player tries to sink the other's fleet of ships by guessing their locations on a grid. With a board size of 10x10 and each cell having 2 possible states (hit or miss), the state space of the game is $2^{100}$, which is very large but still possible to model in a 128-bit integer. The action space is also large, with 100 possible actions (guessing each cell). This makes it an interesting problem for reinforcement learning, as it requires strategic decision-making and can be used to test various RL algorithms.


In [451]:
%load_ext autoreload
%autoreload 2

import logging
from pathlib import Path

import altair as alt
import polars as pl

from game.fleet_placement_methods import PLACEMENT_METHODS
from game.game_logger import GameLogger
from main import run_games_headless

GameLogger.setup(console_level=logging.WARNING)
alt.data_transformers.enable("vegafusion")

# Flags to control which CSV files to generate
MAKE_PRE_ANALYSIS_CSV = False
MAKE_Q_AGENT_TRAIN_CSV = False
MAKE_Q_AGENT_TEST_CSV = False

# Directories for data and images
DATA_DIR = Path("data")
IMG_DIR = Path("img")
DATA_DIR.mkdir(exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

# CSV files
PRE_ANALYSIS_CSV = DATA_DIR / "pre_analysis.csv"
Q_AGENT_TRAIN_CSV = DATA_DIR / "q_agent_train.csv"
Q_AGENT_TEST_CSV = DATA_DIR / "q_agent_test.csv"

# Checkpoint path for Q-agent training
CHECKPOINT_PATH = Path("checkpoints") / "q_agent.pt"

# Config values for charts
 # Default number of games to average over for moving average charts
WINDOW_SIZE = 10
# Number of games to run for each agent and placement method configuration
GAMES_PER_CONFIG = 1000
# List of agent types to evaluate
AGENT_TYPES = [
    "random",
    "hunt",
    "bayes",
    "q-agent",
]
# List of placement methods to evaluate
PLACEMENT_ORDER = list(PLACEMENT_METHODS.keys())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## Baseline Approach

There are multiple ways to approach this problem algorithmically. We will look at three different agents to serve as a baseline for our RL model: a random agent, a "hunt" agent, and a bayesian agent.

### Random Agent

The random agent simply guesses a random valid cell on the board for each turn. That is, the agent cannot repeat guesses and will only select from the remaining cells. This is straightforward to implement and serves as a lower bound for performance.

### Hunt Agent

The hunt agent uses a simple hunt strategy. It starts by randomly guessing cells in a checkerboard pattern (e.g., guessing all the cells where the sum of the row and column indices is even). Once it gets a hit, it switches to a "hunt" mode where it guesses the adjacent cells to try to sink the ship.

### Bayesian Agent

The bayesian agent maintains a probability distribution over the board, representing the likelihood of each cell containing a ship based on the hits and misses observed so far. It updates this distribution after each guess and selects the cell with the highest probability for the next guess.

### Baseline Performance

To evaluate the performance of these baseline agents, we run 10,000 games against a random-guessing opponent and record the win-rate and average number of turns it takes for each agent to win. Whether the player or the agent goes first can affect the outcome so we employ a fair coin flip to determine the starting player. We also test for different ship placement strategies as the ship placement may influence the agents' performance. The results are as follows:


---

## Data Creation


In [452]:
if MAKE_PRE_ANALYSIS_CSV:
    PRE_ANALYSIS_CSV.parent.mkdir(parents=True, exist_ok=True)
    records = []
    configs = [(a, m) for a in AGENT_TYPES for m in PLACEMENT_METHODS]
    n = len(configs)
    for i, (agent_type, method) in enumerate(configs):
        print(f"[{i + 1}/{n}] {agent_type} / {method} {'...':20s}", end="\r")
        for game_id, game in enumerate(
            run_games_headless(
                agent_type, method, GAMES_PER_CONFIG, checkpoint_path=CHECKPOINT_PATH
            )
        ):
            records.append(
                {
                    "agent_type": agent_type,
                    "placement_method": method,
                    "game_id": game_id,
                    **game,
                }
            )
    df = pl.DataFrame(records)
    df.write_csv(PRE_ANALYSIS_CSV)
    print(f"\nSaved {len(df):,} rows to {PRE_ANALYSIS_CSV}")
else:
    df = pl.read_csv(PRE_ANALYSIS_CSV)
    print(f"Loaded {len(df):,} rows from {PRE_ANALYSIS_CSV}")

df_no_q = df.filter(pl.col("agent_type") != "q-agent")
df.show()

Loaded 40,000 rows from data/pre_analysis.csv


agent_type,placement_method,game_id,agent_won,turns,agent_hits,agent_sunk,player_hits,player_sunk
str,str,i64,bool,i64,i64,i64,i64,i64
"""random""","""random""",0,true,94,17,5,16,4
"""random""","""random""",1,false,96,16,4,17,5
"""random""","""random""",2,false,99,16,4,17,5
"""random""","""random""",3,true,97,17,5,14,3
"""random""","""random""",4,false,89,15,3,17,5


In [453]:
summary = (
    df.group_by(["agent_type", "placement_method"])
    .agg(
        pl.col("agent_won").mean().alias("win_rate"),
        pl.col("turns").mean().alias("avg_turns"),
        pl.col("agent_hits").mean().alias("avg_agent_hits"),
        pl.col("player_hits").mean().alias("avg_player_hits"),
        pl.col("agent_sunk").mean().alias("avg_agent_sunk"),
        pl.col("player_sunk").mean().alias("avg_player_sunk"),
        pl.col("game_id").count().alias("n_games"),
    )
    .with_columns(
        (pl.col("win_rate") * 100).round(1).alias("win_rate_pct"),
        pl.col("avg_turns").round(1),
    )
    .sort(["agent_type", "placement_method"])
)
summary.show()

agent_type,placement_method,win_rate,avg_turns,avg_agent_hits,avg_player_hits,avg_agent_sunk,avg_player_sunk,n_games,win_rate_pct
str,str,f64,f64,f64,f64,f64,f64,u32,f64
"""bayes""","""clustered""",1.0,42.3,17.0,7.105,5.0,0.397,1000,100.0
"""bayes""","""cognitive_human""",1.0,42.6,17.0,7.295,5.0,0.463,1000,100.0
"""bayes""","""corners""",1.0,52.1,17.0,8.895,5.0,0.712,1000,100.0
"""bayes""","""dense_center""",1.0,34.8,17.0,5.858,5.0,0.265,1000,100.0
"""bayes""","""diagonal""",1.0,45.6,17.0,7.761,5.0,0.495,1000,100.0


---

## Altair Helpers


In [454]:
def chart_title(text: str) -> alt.TitleParams:
    return alt.TitleParams(text, fontSize=14, fontWeight="normal", anchor="middle")


# Heatmap: agent win rate by agent type x player placement method
def make_heatmap(df: pl.DataFrame, title: str) -> alt.LayerChart | alt.FacetChart:
    heatmap = (
        alt.Chart(df)
        .mark_rect()
        .encode(
            x=alt.X(
                "placement_method:N",
                title="Player Placement Method",
                axis=alt.Axis(labelAngle=-35),
            ),
            y=alt.Y("agent_type:N", title="Agent Type"),
            color=alt.Color(
                "avg_turns:Q",
                title="Average Turns",
                scale=alt.Scale(scheme="blues", domain=[100, 30]),
            ),
            tooltip=[
                alt.Tooltip("agent_type:N", title="Agent"),
                alt.Tooltip("placement_method:N", title="Placement"),
                alt.Tooltip("win_rate_pct:Q", title="Win Rate (%)", format=".1f"),
                alt.Tooltip("avg_turns:Q", title="Avg Turns", format=".1f"),
            ],
        )
        .properties(
            title=chart_title(title),
            width=500,
            height=150,
        )
    )

    labels = (
        alt.Chart(df)
        .mark_text(fontSize=11)
        .encode(
            x=alt.X("placement_method:N"),
            y=alt.Y("agent_type:N"),
            text=alt.Text("avg_turns:Q", format=".0f"),
            color=alt.condition(
                alt.datum.avg_turns < 60,
                alt.value("white"),
                alt.value("black"),
            ),
        )
    )

    return heatmap + labels

In [455]:
# Bar chart: avg turns by agent type (collapsed across placement methods)
def make_bar_chart(df: pl.DataFrame, title: str) -> alt.Chart:
    turns_by_agent = df.group_by("agent_type").agg(
        pl.col("turns").mean().round(2).alias("avg_turns"),
        pl.col("turns").std().alias("std_turns"),
    )

    return (
        alt.Chart(turns_by_agent)
        .mark_bar()
        .encode(
            x=alt.X("agent_type:N", title="Agent Type"),
            y=alt.Y("avg_turns:Q", title="Average Turns"),
            color=alt.Color("agent_type:N", legend=None),
            tooltip=[
                alt.Tooltip("agent_type:N", title="Agent"),
                alt.Tooltip("avg_turns:Q", title="Avg Turns", format=".2f"),
                alt.Tooltip("std_turns:Q", title="Std Dev", format=".2f"),
            ],
        )
        .properties(
            title=chart_title(title),
            width=300,
            height=250,
        )
    )

In [456]:
# Grouped bars: avg agent hits vs player hits by agent type
def make_grouped_bar_chart(df: pl.DataFrame, title: str) -> alt.Chart:
    hits_long = pl.concat(
        [
            df.select(
                [
                    "agent_type",
                    pl.col("agent_hits").alias("hits"),
                    pl.lit("Agent").alias("side"),
                ]
            ),
            df.select(
                [
                    "agent_type",
                    pl.col("player_hits").alias("hits"),
                    pl.lit("Player (random)").alias("side"),
                ]
            ),
        ]
    )

    hits_summary = hits_long.group_by(["agent_type", "side"]).agg(
        pl.col("hits").mean().alias("avg_hits")
    )

    return (
        alt.Chart(hits_summary)
        .mark_bar()
        .encode(
            x=alt.X("agent_type:N", title="Agent Type"),
            y=alt.Y("avg_hits:Q", title="Average Hits per Game"),
            xOffset=alt.XOffset("side:N"),
            color=alt.Color("side:N", title="Side"),
            tooltip=[
                alt.Tooltip("agent_type:N", title="Agent"),
                alt.Tooltip("side:N", title="Side"),
                alt.Tooltip("avg_hits:Q", title="Avg Hits", format=".1f"),
            ],
        )
        .properties(
            title=chart_title(title),
            width=350,
            height=250,
        )
    )

In [457]:
summary_no_q = summary.filter(pl.col("agent_type") != "q-agent")
c = make_heatmap(summary, "Agent Performance by Type and Player Placement Method")
c.save(IMG_DIR / "win_rate_heatmap.png", ppi=300, scale_factor=2)
c.show()

alt.LayerChart(...)

In [458]:
c = make_bar_chart(df, "Average Turns by Agent Type (All Placements)")
c.save(IMG_DIR / "avg_turns_by_agent.png", ppi=300, scale_factor=2)
c.show()

alt.Chart(...)

In [459]:
c = make_grouped_bar_chart(df, "Average Hits per Game: Agent vs Player (Random)")
c.save(IMG_DIR / "avg_hits_by_agent.png", ppi=300, scale_factor=2)
c.show()

alt.Chart(...)

In [460]:
# get the winrate for each agent type across all placement methods
win_rates = (
    df.group_by("agent_type")
    .agg(pl.col("agent_won").mean().alias("win_rate"))
    .with_columns((pl.col("win_rate") * 100).round(1).alias("win_rate_pct"))
)
win_rates.show()

agent_type,win_rate,win_rate_pct
str,f64,f64
"""q-agent""",0.9991,99.9
"""bayes""",1.0,100.0
"""random""",0.5036,50.4
"""hunt""",0.9998,100.0


---

## Baseline Analysis

The random agent performs as expected, with a win rate around 50% and an average of around 93 turns to win. The hunt agent performs significantly better, with a win rate of near 100% and between 44 and 58 turns to win, depending on the ship placement strategy. The bayesian agent performs the best, also with a win rate of near 100%, and an average between 35 and 52 turns to win. It is notable that the bayesian agent struggled the most with the strategies which spread ships apart (corners, edges, and spread). This makes sense given that the bayesian agent relies on clustering of hits to update its probability distribution, so when ships are spread apart it has less information to work with.

Overall, these metrics provide a strong baseline for evaluating the performance of our RL agent. Sophisticated algorithms and heuristics can easily reach fewer than 50 turns to win, so we will aim to beat that threshold with our RL agent.


---

# Q-Learning RL Agent

We implemented a Q-learning agent to learn how to play Battleship. This agent performs Q-learning on a CNN-based state representation of the boart. The state is represented as a 10x10x3 tensor, where the first channel represents the bayesian probability of a ship being present, the second channel represents the hits, and the third channel represents the misses. The action space is represented as a 100-dimensional vector, where each element corresponds to a cell on the board and contains the Q-value for that action. We also encode which ships have been sunk as a global feature. Finally, we using a legal mask to ensure that the agent only selects from valid actions (i.e., cells that have not been guessed yet).

During the training process, we used performed a supervised pre-training phase where the agent learned to predict the bayesian probability distribution over the board, given the hits and misses observed so far. We used the aforementioned Bayesian agent to label the training data. This helped the agent learn a useful representation of the state space before we started the Q-learning phase. After pre-training, pre-populated the replay buffer used in Q-learning with games played by the bayesian agent. This is a necessary step to ensure that the agent does not immediately forget the useful information it learned during pre-training when we start the Q-learning phase. Finally, we trained the agent using the standard Q-learning algorithm with epsilon-greedy exploration and a replay buffer.

The final training pipeline looks like this:

$$
\text{Pre-training} \rightarrow \text{Replay Buffer Initialization} \rightarrow \text{Q-learning Training}
$$


---

## Charting Utils


In [461]:
def make_chart(
    df: pl.DataFrame,
    y_col: str,
    window_size: int = WINDOW_SIZE,
    arg_col: str = "step",
    scale_x: bool = True,
    stroke: float = 2,
) -> alt.Chart | alt.LayerChart | alt.FacetChart:
    df = df.with_columns(
        pl.col(y_col).rolling_mean(window_size=window_size).alias(f"{y_col}_rolling"),
    )
    window_size = max(window_size, 1)
    # Chart size calculations
    x_max = df[arg_col].max() * 1.05
    x_axis = alt.Axis()
    if scale_x:
        x_axis = alt.Axis(labelExpr="datum.value / 1000 + 'k'")
    x_scale = alt.Scale(domainMax=x_max)

    def format_label(label: str) -> str:
        def _cap(r: str) -> str:
            return " ".join([s.capitalize() for s in label.split(r)])

        if "_" in label:
            return _cap("_")
        elif "-" in label:
            return _cap("-")
        return label.capitalize()

    y_col_label = format_label(y_col)
    x_col_label = format_label(arg_col)

    # Define color scale for lines and rules
    series_name = "Rolling avg" if window_size > 1 else "Avg"
    y_label = (
        f"{y_col_label} ({window_size}-{x_col_label} rolling avg)"
        if window_size > 1
        else y_col_label
    )
    # Rolling average line
    line = (
        alt.Chart(df.with_columns(pl.lit(series_name).alias("series")))
        .mark_line(strokeWidth=stroke, color="steelblue")
        .encode(
            x=alt.X(arg_col, title=x_col_label, axis=x_axis, scale=x_scale),
            y=alt.Y(
                f"{y_col}_rolling",
                title=y_label,
            ),
        )
        .properties(
            title=f"{y_col_label} v {x_col_label}",
            width=500,
            height=300,
        )
    )

    return line

In [462]:
def make_dual_line(
    df,
    col1,
    col2,
    y_title,
    chart_title,
    window=WINDOW_SIZE,
    stroke: float = 1,
):
    """Rolling avg line chart comparing agent vs player on a single metric."""
    rolling = (
        df.select(["game", col1, col2])
        .with_columns(
            [
                pl.col(col1).rolling_mean(window_size=window).alias(f"{col1}_r"),
                pl.col(col2).rolling_mean(window_size=window).alias(f"{col2}_r"),
            ]
        )
        .select(["game", f"{col1}_r", f"{col2}_r"])
        .unpivot(
            index="game",
            on=[f"{col1}_r", f"{col2}_r"],
            variable_name="side",
            value_name="value",
        )
        .with_columns(
            pl.col("side").replace({f"{col1}_r": "Agent", f"{col2}_r": "Player"})
        )
    )
    return (
        alt.Chart(rolling)
        .mark_line(strokeWidth=stroke)
        .encode(
            x=alt.X("game:Q", title="Game"),
            y=alt.Y("value:Q", title=f"{y_title} ({window}-game rolling avg)"),
            color=alt.Color(
                "side:N",
                scale=alt.Scale(
                    domain=["Agent", "Player"], range=["steelblue", "coral"]
                ),
                legend=alt.Legend(title=""),
            ),
        )
        .properties(title=chart_title, width=500, height=300)
    )

## Rules for Baseline Comparison


In [463]:
BAYESIAN_BASELINE_TURNS = summary.filter(pl.col("agent_type") == "bayes")[
    "avg_turns"
].mean()
HUNT_BASELINE_TURNS = summary.filter(pl.col("agent_type") == "hunt")["avg_turns"].mean()

bayes_baseline_rule = (
    alt.Chart(
        pl.DataFrame({"y": [BAYESIAN_BASELINE_TURNS], "series": ["Bayes target"]})
    )
    .mark_rule(strokeDash=[4, 2], color="red")
    .encode(
        y=alt.Y("y:Q"),
        color=alt.Color(
            "series:N",
            scale=alt.Scale(domain=["Bayes target"], range=["red"]),
            legend=alt.Legend(title=""),
        ),
    )
)

hunt_baseline_rule = (
    alt.Chart(pl.DataFrame({"y": [HUNT_BASELINE_TURNS], "series": ["Hunt target"]}))
    .mark_rule(strokeDash=[4, 2], color="orange")
    .encode(
        y=alt.Y("y:Q"),
        color=alt.Color(
            "series:N",
            scale=alt.Scale(domain=["Hunt target"], range=["orange"]),
            legend=alt.Legend(title=""),
        ),
    )
)

---

# Q-Training Data


In [464]:
latest_q_train_log = max(
    Path("training/logs").glob("q_train_*.log"), key=lambda x: x.stat().st_mtime
)
q_train_log_file = latest_q_train_log

if MAKE_Q_AGENT_TRAIN_CSV:
    with open(q_train_log_file, "r") as f:
        lines = [line.split("INFO")[1].strip() for line in f if "ep=" in line]
        data = []
        for line in lines:
            parts = line.split()
            ep = int(parts[0].split("=")[1])
            eps = float(parts[1].split("=")[1])
            steps = int(parts[2].split("=")[1])
            mean_turns = float(parts[3].split("=")[1])
            data.append((ep, eps, steps, mean_turns))
        q_train_df = pl.DataFrame(
            data, schema=["episode", "epsilon", "steps", "mean_turns"], orient="row"
        )
    q_train_df.write_csv(Q_AGENT_TRAIN_CSV)

# Q-Agent Data


In [465]:
latest_q_agent_log = max(
    Path("game/logs").glob("game_*.log"), key=lambda x: x.stat().st_mtime
)
q_agent_log_file = latest_q_agent_log
df_schema = {
    "game": pl.Int64,
    "agent_won": pl.Boolean,
    "player_won": pl.Boolean,
    "agent_sunk": pl.Int64,
    "player_sunk": pl.Int64,
    "agent_hit": pl.Int64,
    "player_hit": pl.Int64,
    "agent_hit_diff": pl.Float64,
    "agent_sink_diff": pl.Float64,
    "turns": pl.Int64,
}
df = pl.DataFrame(schema=df_schema)

if MAKE_Q_AGENT_TEST_CSV:
    with open(q_agent_log_file, "r") as f:
        lines = [line.split("INFO")[1].strip() for line in f if "INFO" in line]
        i = 0
        game_num = 0
        while i < len(lines):
            game_data = {}

            line = lines[i]
            if not line.startswith("GAME OVER"):
                i += 1
                continue

            game_num += 1
            parts = line.split("|")
            winner_str = parts[0].split("Winner:")[1].strip()
            turns = int(parts[1].split("Turns:")[1].strip())
            game_data["agent_won"] = winner_str == "Agent"
            game_data["player_won"] = winner_str == "Player"
            game_data["turns"] = turns
            game_data["game"] = game_num
            i += 1

            line = lines[i]
            if not line.strip().startswith("Player"):
                i += 1
                continue

            parts = line.split("—")[1].split(",")
            sunk = int(parts[0].strip().split(" ")[1].split(",")[0].strip())
            hit = int(parts[1].strip().split(" ")[1].strip())
            game_data["player_sunk"] = sunk
            game_data["player_hit"] = hit
            i += 1

            line = lines[i]
            if not line.strip().startswith("Agent"):
                i += 1
                continue

            parts = line.split("—")[1].split(",")
            sunk = int(parts[0].strip().split(" ")[1].split(",")[0].strip())
            hit = int(parts[1].strip().split(" ")[1].strip())
            game_data["agent_sunk"] = sunk
            game_data["agent_hit"] = hit
            i += 1

            # Normalized differences for hits and sunk (max hits = 17, max sunk = 5)
            agent_hit_diff = (game_data["agent_hit"] - game_data["player_hit"]) / 17
            agent_sink_diff = (game_data["agent_sunk"] - game_data["player_sunk"]) / 5

            df = df.extend(
                pl.DataFrame(
                    schema=df_schema,
                    data={
                        "game": game_data["game"],
                        "agent_won": game_data["agent_won"],
                        "player_won": game_data["player_won"],
                        "agent_sunk": game_data["agent_sunk"],
                        "player_sunk": game_data["player_sunk"],
                        "agent_hit": game_data["agent_hit"],
                        "player_hit": game_data["player_hit"],
                        "agent_hit_diff": agent_hit_diff,
                        "agent_sink_diff": agent_sink_diff,
                        "turns": game_data["turns"],
                    },
                )
            )
    df.write_csv(Q_AGENT_TEST_CSV)

---

# Q-Training Analysis


In [466]:
q_learning_df = pl.read_csv(Q_AGENT_TRAIN_CSV)
q_learning_df.show()

episode,epsilon,steps,mean_turns
i64,f64,i64,f64
100,0.297,6837,65.9
200,0.2941,14098,85.0
300,0.2911,22575,78.8
400,0.2882,30252,73.7
500,0.2854,36644,66.4


In [467]:
# Summary Stats
print(f"Total episodes:  {q_learning_df['episode'].max()}")
print(f"Total steps:     {q_learning_df['steps'].max()}")
print(f"Best mean turns: {q_learning_df['mean_turns'].min():.1f}")
print(f"Hunt baseline:   {HUNT_BASELINE_TURNS:.1f}")
print(f"Bayes baseline:  {BAYESIAN_BASELINE_TURNS:.1f}")

mean_turns = make_chart(q_learning_df, "mean_turns", arg_col="episode")
mean_turns += bayes_baseline_rule + hunt_baseline_rule

title = alt.TitleParams("Q-Learning Training - Mean Turns", anchor="middle")
c = mean_turns.resolve_scale(color="independent").properties(title=title)
c.save(IMG_DIR / "mean_turns.png", ppi=300, scale_factor=2)
c.show()

Total episodes:  20000
Total steps:     1004115
Best mean turns: 44.5
Hunt baseline:   51.2
Bayes baseline:  45.9


alt.LayerChart(...)

---

# Q-Agent Testing Analysis


In [468]:
q_agent_df = pl.read_csv(Q_AGENT_TEST_CSV)
q_agent_df.show()

game,agent_won,player_won,agent_sunk,player_sunk,agent_hit,player_hit,agent_hit_diff,agent_sink_diff,turns
i64,bool,bool,i64,i64,i64,i64,f64,f64,i64
1,false,true,4,5,15,17,-0.117647,-0.2,53
2,false,true,4,5,15,17,-0.117647,-0.2,43
3,true,false,5,4,17,14,0.176471,0.2,41
4,false,true,3,5,14,17,-0.176471,-0.4,29
5,false,true,2,5,8,17,-0.529412,-0.6,36


In [469]:
GAME_WINDOW = 100
q_agent_df["agent_won"].mean()
n_games = len(q_agent_df)
agent_wr = float(q_agent_df["agent_won"].mean()) * 100  # type: ignore
player_wr = float(q_agent_df["player_won"].mean()) * 100  # type: ignore
avg_turns = float(q_agent_df["turns"].mean())  # type: ignore
avg_agent_hits = float(q_agent_df["agent_hit"].mean())  # type: ignore
avg_player_hits = float(q_agent_df["player_hit"].mean())  # type: ignore
avg_agent_sunk = float(q_agent_df["agent_sunk"].mean())  # type: ignore
avg_player_sunk = float(q_agent_df["player_sunk"].mean())  # type: ignore

print(f"Games:            {n_games}")
print(f"Agent win rate:   {agent_wr:.1f}%  |  Player win rate: {player_wr:.1f}%")
print(f"Avg turns/game:   {avg_turns:.1f}")
print(
    f"Avg agent hits:   {avg_agent_hits:.1f}  |  Avg player hits: {avg_player_hits:.1f}"
)
print(
    f"Avg agent sinks:  {avg_agent_sunk:.1f}  |  Avg player sinks: {avg_player_sunk:.1f}"
)

rolling = q_agent_df.with_columns(
    [
        (pl.col("agent_won").cast(pl.Float64) * 100)
        .rolling_mean(window_size=GAME_WINDOW)
        .alias("agent_win_rate")
    ]
)

win_long = (
    rolling.select(["game", "agent_win_rate"])
    .unpivot(
        index="game",
        on=["agent_win_rate"],
        variable_name="side",
        value_name="win_rate",
    )
    .with_columns(pl.col("side").replace({"agent_win_rate": "Agent"}))
)

rolling_chart = make_chart(
    win_long,
    "win_rate",
    arg_col="game",
    window_size=GAME_WINDOW,
    stroke=0.5,
)

color_scale = alt.Scale(domain=["Agent", "Bayes"], range=["steelblue", "coral"])

win_summary = pl.DataFrame(
    {"side": ["Agent", "Bayes"], "win_rate": [agent_wr, player_wr]}
)
bar = (
    alt.Chart(win_summary)
    .mark_bar(size=50)
    .encode(
        x=alt.X("side:N", title=""),
        y=alt.Y("win_rate:Q", title="Win Rate (%)", scale=alt.Scale(domain=[0, 100])),
        color=alt.Color("side:N", scale=color_scale, legend=None),
    )
    .properties(title="Overall Win Rate", width=180, height=300)
)
bar_text = bar.mark_text(dy=-12, fontSize=13).encode(
    text=alt.Text("win_rate:Q", format=".1f")
)

title = alt.TitleParams("Q-Agent - Win Rates", anchor="middle", fontSize=18)
c = (
    ((rolling_chart) | (bar + bar_text))
    .resolve_scale(y="independent", color="independent")
    .properties(title=title)
)
c.save(IMG_DIR / "q_agent_win_rates.png", ppi=300, scale_factor=2)
c.show()

Games:            10000
Agent win rate:   52.1%  |  Player win rate: 47.9%
Avg turns/game:   39.2
Avg agent hits:   15.2  |  Avg player hits: 14.8
Avg agent sinks:  4.3  |  Avg player sinks: 4.1


alt.HConcatChart(...)

In [470]:
STAT_WINDOW = 100

zero_rule = (
    alt.Chart(pl.DataFrame({"y": [0]}))
    .mark_rule(strokeDash=[4, 4], color="red")
    .encode(y="y:Q")
)

turns_chart = make_chart(
    q_agent_df,
    "turns",
    arg_col="game",
    window_size=STAT_WINDOW,
    stroke=0.5,
)
hits_chart = (
    make_chart(
        q_agent_df,
        "agent_hit_diff",
        arg_col="game",
        window_size=STAT_WINDOW,
        stroke=0.5,
    )
    + zero_rule
)
sinks_chart = (
    make_chart(
        q_agent_df,
        "agent_sink_diff",
        arg_col="game",
        window_size=STAT_WINDOW,
        stroke=0.5,
    )
    + zero_rule
)

c = turns_chart
c.save(IMG_DIR / "q_agent_turns.png", ppi=300, scale_factor=2)
c.show()

title = alt.TitleParams("Q-Agent - Targeting Statistics", anchor="middle", fontSize=18)
c = (hits_chart | sinks_chart).resolve_scale(y="independent").properties(title=title)
c.save(IMG_DIR / "q_agent_targeting_stats.png", ppi=300, scale_factor=2)
c.show()

alt.Chart(...)

alt.HConcatChart(...)

In [471]:
# 1. Turns per game distribution
turns_hist = (
    alt.Chart(q_agent_df)
    .mark_bar()
    .encode(
        x=alt.X("turns:Q", title="Total Turns", bin=alt.Bin(step=5)),
        y=alt.Y("count():Q", title="Games"),
        color=alt.value("steelblue"),
    )
    .properties(title="Turns per Game Distribution", width=300, height=250)
)

# 2. Agent turns on wins vs losses (strip + mean tick)
outcome_df = q_agent_df.with_columns(
    pl.when(pl.col("agent_won"))
    .then(pl.lit("Agent Win"))
    .otherwise(pl.lit("Agent Loss"))
    .alias("outcome")
)
strip = (
    alt.Chart(outcome_df)
    .mark_circle(opacity=0.4, size=30)
    .encode(
        x=alt.X("outcome:N", title=""),
        y=alt.Y("turns:Q", title="Total Turns"),
        color=alt.Color(
            "outcome:N",
            scale=alt.Scale(
                domain=["Agent Win", "Agent Loss"], range=["steelblue", "coral"]
            ),
            legend=None,
        ),
    )
    .properties(title="Agent Turns: Wins vs Losses", width=220, height=250)
)
mean_tick = (
    alt.Chart(outcome_df)
    .mark_tick(thickness=3, size=40, color="black")
    .encode(x=alt.X("outcome:N"), y=alt.Y("mean(turns):Q"))
)
turns_outcome = strip + mean_tick

# 3. Hit accuracy per game (hits / total turns, rolling)
acc_df = q_agent_df.with_columns(
    [
        (pl.col("agent_hit") / pl.col("turns") * 100).alias("agent_acc"),
        (pl.col("player_hit") / pl.col("turns") * 100).alias("player_acc"),
    ]
)
acc_chart = make_dual_line(
    acc_df,
    "agent_acc",
    "player_acc",
    "Hit Accuracy (%)",
    "Hit Accuracy (hits/turns %)",
    window=STAT_WINDOW,
    stroke=0.5,
)

title = alt.TitleParams("Q-Agent - Additional Analysis", anchor="middle", fontSize=18)
c = (
    (turns_hist | turns_outcome | acc_chart)
    .resolve_scale(y="independent", color="independent")
    .properties(title=title)
)
c.save(IMG_DIR / "q_agent_additional_analysis.png", ppi=300, scale_factor=2)
c.show()

alt.HConcatChart(...)

---

# Conclusion

The Q-learning agent was able to learn a policy that outperforms the baselines across multiple ship placement strategies. The agent was able to achieve an average of around 40 turns to win, which is a significant improvement over the baselines. The agent also learned to adapt its strategy based on the ship placement, performing better on strategies that the bayesian agent performed poorly on, such as edges and corners. This suggests that the agent was able to learn a more sophisticated strategy that takes into account the spatial distribution of hits and misses on the board.
